In [ ]:
!pip install face_recognition
!pip install opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566166 sha256=a9795824c859a94b12d8e09a7a9407e5f96d357cbeca3c83102c3f3a5fab100c
  Stored in directory: /root/.cache/pip/wheels/8f/47/c8/f44c5aebb7507f7c8a2c0bd23151d732d0f0bd6884ad4ac635
Successfully built face-recognition-models


In [ ]:
import face_recognition
import cv2
import numpy as np
import pandas as pd
from datetime import datetime
from google.colab import files
import os

In [ ]:
  os.makedirs("known_faces", exist_ok=True)

In [ ]:
print("Upload 8 student images")

uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, "known_faces/" + filename)

Upload 8 student images


Saving divya.jpeg to divya.jpeg
Saving mayank.jpeg to mayank.jpeg
Saving pratham.jpeg to pratham.jpeg
Saving riya  - Copy.jpeg to riya  - Copy.jpeg
Saving roshini.jpeg to roshini.jpeg
Saving shravani.jpeg to shravani.jpeg
Saving shriya - Copy.jpeg to shriya - Copy.jpeg
Saving sujal.jpeg to sujal.jpeg


In [ ]:
known_face_encodings = []
known_face_names = []

for filename in os.listdir("known_faces"):

    image = face_recognition.load_image_file("known_faces/" + filename)
    encodings = face_recognition.face_encodings(image)

    if len(encodings) > 0:

        known_face_encodings.append(encodings[0])
        known_face_names.append(os.path.splitext(filename)[0])

print("Registered Students:", known_face_names)

Registered Students: ['shravani', 'sujal', 'riya  - Copy', 'mayank', 'roshini', 'divya', 'shriya - Copy', 'pratham']


In [ ]:
subjects = ["Maths","Science","English"]

In [ ]:
print("Upload subject group photos")

uploaded = files.upload()

Upload subject group photos


Saving english.jpeg to english.jpeg
Saving math - Copy.jpeg to math - Copy.jpeg
Saving science.jpeg to science.jpeg


In [ ]:
attendance = pd.DataFrame({
    "Name": known_face_names,
    "Maths": ["Absent"]*len(known_face_names),
    "Science": ["Absent"]*len(known_face_names),
    "English": ["Absent"]*len(known_face_names)
})

In [ ]:
correct_predictions = 0
total_predictions = 0

for filename in uploaded.keys():

    subject = filename.split(".")[0].capitalize()

    image = face_recognition.load_image_file(filename)

    face_locations = face_recognition.face_locations(image)
    face_encodings = face_recognition.face_encodings(image, face_locations)

    detected_students = []

    for face_encoding in face_encodings:

        matches = face_recognition.compare_faces(known_face_encodings, face_encoding)

        name = "Unknown"

        face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)

        best_match_index = np.argmin(face_distances)

        if matches[best_match_index]:

            name = known_face_names[best_match_index]

            detected_students.append(name)

            correct_predictions += 1

        total_predictions += 1

    print(subject,"Detected:", detected_students)

    for student in detected_students:

        attendance.loc[attendance["Name"] == student, subject] = "Present"

English Detected: ['mayank', 'divya', 'shravani', 'riya  - Copy']
Math - copy Detected: ['shriya - Copy', 'sujal', 'divya', 'riya  - Copy']
Science Detected: ['shriya - Copy', 'mayank', 'shravani', 'sujal', 'pratham']


In [ ]:
print("\nAttendance Table")
print(attendance)


Attendance Table
            Name   Maths  Science  English Math - copy
0       shravani  Absent  Present  Present         NaN
1          sujal  Absent  Present   Absent     Present
2   riya  - Copy  Absent   Absent  Present     Present
3         mayank  Absent  Present  Present         NaN
4        roshini  Absent   Absent   Absent         NaN
5          divya  Absent   Absent  Present     Present
6  shriya - Copy  Absent  Present   Absent     Present
7        pratham  Absent  Present   Absent         NaN


In [ ]:
if total_predictions > 0:

    accuracy = (correct_predictions / total_predictions) * 100

    print("Face Recognition Accuracy:", round(accuracy,2),"%")

Face Recognition Accuracy: 92.86 %


In [ ]:
attendance.to_excel("attendance_report.xlsx", index=False)

print("Attendance Excel File Created")

Attendance Excel File Created


In [ ]:
files.download("attendance_report.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>